In [4]:
import pandas as pd
import numpy as np
import re

from pathlib import Path
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

Libraries imported successfully.


In [5]:
RAW_PATH = Path("../data/raw/customer_support_tickets.csv")

PROCESSED_DIR = Path("../data/processed")
SPLITS_DIR = Path("../data/splits")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

print("Directories ready.")


Directories ready.


In [6]:
#Load raw data
df = pd.read_csv(RAW_PATH)

print(f"Raw dataset shape: {df.shape}")

df.head()

Raw dataset shape: (28587, 16)


,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51,Security,Outage,Disruption,Data Breach,NaN,NaN,NaN,NaN
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51,Account,Disruption,Outage,IT,Tech Support,NaN,NaN,NaN
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51,Product,Feature,Tech Support,NaN,NaN,NaN,NaN,NaN
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51,Billing,Payment,Account,Documentation,Feedback,NaN,NaN,NaN
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51,Product,Feature,Feedback,Tech Support,NaN,NaN,NaN,NaN


In [7]:
#keeping only english tickets 
print("Language distribution before filtering:")

df["language"].value_counts()

df_en = df[df["language"] == "en"].copy()

print(f"English dataset shape: {df_en.shape}")

df_en["language"].value_counts()


Language distribution before filtering:
English dataset shape: (16338, 16)


language
en    16338
Name: count, dtype: int64

In [8]:
#Select relevant columns
MODEL_COLUMNS = [
    "subject",
    "body",
    "queue",
    "priority",
    "type"
]

df_model = df_en[MODEL_COLUMNS].copy()

print(f"Modeling dataset shape: {df_model.shape}")

df_model.head()

Modeling dataset shape: (16338, 5)


,subject,body,queue,priority,type
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",Technical Support,high,Incident
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Returns and Exchanges,medium,Request
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Billing and Payments,low,Request
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Sales and Pre-Sales,medium,Problem
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Technical Support,high,Request


In [9]:
#handle missing values
df_model["subject"] = df_model["subject"].fillna("").astype(str)
df_model["body"] = df_model["body"].fillna("").astype(str)

print("Missing values after text handling:")

df_model[["subject", "body"]].isnull().sum()

Missing values after text handling:


subject    0
body       0
dtype: int64

In [10]:
df_model["text"] = (
    df_model["subject"].str.strip()
    + " "
    + df_model["body"].str.strip()
).str.strip()

df_model[["subject", "body", "text"]].head()

,subject,body,text
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Account Disruption Dear Customer Support Team,..."
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Query About Smart Home System Integration Feat...
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Inquiry Regarding Invoice Details Dear Custome...
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Question About Marketing Agency Software Compa...
5,Feature Query,"Dear Customer Support,\n\nI hope this message ...","Feature Query Dear Customer Support,\n\nI hope..."


In [11]:
#Remove records without usable text
before_empty_removal = len(df_model)

df_model = df_model[df_model["text"].str.strip() != ""].copy()

after_empty_removal = len(df_model)

print(f"Removed records: {before_empty_removal - after_empty_removal}")
print(f"Remaining records: {after_empty_removal}")

Removed records: 0
Remaining records: 16338


In [12]:
#cleaning the text
def clean_text(text):
    text = text.lower()

    # Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


In [ ]:
df_model["clean_text"] = df_model["text"].apply(clean_text)
df_model[["text", "clean_text"]].head()

,text,clean_text
1,"Account Disruption Dear Customer Support Team,...","account disruption dear customer support team,..."
2,Query About Smart Home System Integration Feat...,query about smart home system integration feat...
3,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...
4,Question About Marketing Agency Software Compa...,question about marketing agency software compa...
5,"Feature Query Dear Customer Support,\n\nI hope...","feature query dear customer support,\n\ni hope..."


In [16]:
#checking duplicates
duplicates_before = df_model["clean_text"].duplicated().sum()
print(f"Duplicate cleaned tickets: {duplicates_before:,}")


df_model = df_model.drop_duplicates(
    subset=["clean_text"]
).copy()

print(f"Dataset shape after duplicate removal: {df_model.shape}")

Duplicate cleaned tickets: 0
Dataset shape after duplicate removal: (16338, 7)


In [17]:
#Check target values
print("Missing queue labels:")
print(df_model["queue"].isnull().sum())

print("\nNumber of queue classes:")
print(df_model["queue"].nunique())

Missing queue labels:
0

Number of queue classes:
10


In [19]:
#Remove missing targets
df_model = df_model.dropna(subset=["queue"]).copy()
print(f"Dataset shape after target validation: {df_model.shape}")

Dataset shape after target validation: (16338, 7)


In [18]:
#Inspect class distribution
queue_distribution = df_model["queue"].value_counts()
queue_distribution

queue
Technical Support                  4737
Product Support                    3073
Customer Service                   2410
IT Support                         1942
Billing and Payments               1595
Returns and Exchanges               820
Service Outages and Maintenance     664
Sales and Pre-Sales                 513
Human Resources                     348
General Inquiry                     236
Name: count, dtype: int64

In [22]:
#Train/validation/test split
#70% 15% 15%
X = df_model["clean_text"]
y = df_model["queue"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))


split_distribution = pd.DataFrame({
    "Full Dataset": y.value_counts(normalize=True),
    "Train": y_train.value_counts(normalize=True),
    "Validation": y_val.value_counts(normalize=True),
    "Test": y_test.value_counts(normalize=True)
}).fillna(0)

split_distribution

Training samples: 11436
Validation samples: 2451
Test samples: 2451


,Full Dataset,Train,Validation,Test
queue,,,,
Technical Support,0.289938,0.289962,0.290086,0.289678
Product Support,0.188089,0.188090,0.188086,0.188086
Customer Service,0.147509,0.147517,0.147695,0.147287
IT Support,0.118864,0.118835,0.118727,0.119135
Billing and Payments,0.097625,0.097587,0.097919,0.097511
Returns and Exchanges,0.050190,0.050192,0.050184,0.050184
Service Outages and Maintenance,0.040641,0.040661,0.040392,0.040800
Sales and Pre-Sales,0.031399,0.031392,0.031416,0.031416
Human Resources,0.021300,0.021336,0.021216,0.021216


In [23]:
#final cleaned dataset

train_df = df_model.loc[X_train.index].copy()
val_df = df_model.loc[X_val.index].copy()
test_df = df_model.loc[X_test.index].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (11436, 7)
Validation: (2451, 7)
Test: (2451, 7)


In [24]:
train_df.to_csv(
    SPLITS_DIR / "train.csv",
    index=False
)

val_df.to_csv(
    SPLITS_DIR / "validation.csv",
    index=False
)

test_df.to_csv(
    SPLITS_DIR / "test.csv",
    index=False
)

print("Train, validation, and test datasets saved.")

Train, validation, and test datasets saved.


In [25]:
processed_path = PROCESSED_DIR / "customer_support_tickets_en.csv"

df_model.to_csv(
    processed_path,
    index=False
)

print(f"Processed dataset saved to: {processed_path}")

Processed dataset saved to: ..\data\processed\customer_support_tickets_en.csv


In [27]:
print("Final processed dataset")
print("-" * 40)

print(f"Total records: {len(df_model):,}")
print(f"Number of queue classes: {df_model['queue'].nunique()}")
print(f"Training records: {len(train_df):,}")
print(f"Validation records: {len(val_df):,}")
print(f"Test records: {len(test_df):,}")

print("\nMissing values:")
print(df_model[["clean_text", "queue"]].isnull().sum())

Final processed dataset
----------------------------------------
Total records: 16,338
Number of queue classes: 10
Training records: 11,436
Validation records: 2,451
Test records: 2,451

Missing values:
clean_text    0
queue         0
dtype: int64
